# 📅 2026-07-03 개발 노트 : 마이페이지 회원 기능 완성 (찜 / 취향 설정 / 회원정보 수정 / 탈퇴 / 사이드바)## 🎯 오늘의 목표 — "다 해보자" (회원 기능 전부 + 사이드바로 마무리)- [x] 찜하기 전체 (로컬 → DB, 로그인 전용)- [x] 회원정보 수정 (온보딩 재활용)- [x] 취향 설정 (선호 장르 + 지표별 점수) ⭐ 핵심- [x] 회원 탈퇴 (익명화, PIPA/GDPR)- [x] 마이페이지 사이드바 레이아웃- [x] Project B(데이터 파이프라인) 진단 — 별도 세션에서 진행 중> 이번 세션에서 마이페이지 회원 기능을 통째로 완성. 메모리 원칙("회원 기능 완성 → 배포 → 마케팅")의 '회원 기능 완성' 지점 도달.

## ⭐ 1. 찜하기 — 로컬 → DB (로그인 전용)**문제:** 기존 favorites가 zustand persist(localStorage)에만 저장 → 로그아웃/기기변경 시 날아감, 서버가 취향 데이터로 못 씀.**방향 (A안 확정):** 찜 = 로그인 기능. 비로그인은 "로그인 후 이용" 안내(거절). 머지 로직 없이 단순.**구현:**- Django `Favorite` 모델 (user, app_id, created_at, UniqueConstraint(user,app_id))- API: `FavoriteToggleView`(토글, {favorited} 반환) + `FavoriteListView`(목록+app_ids, store 동기화용)- api.ts: `toggleFavoriteApi`, `getFavorites`- **store 개조 (방법 1: store를 DB의 거울로)**: toggleFavorite에 낙관적 업데이트+DB반영+실패 롤백 / loadFavorites(로그인 시 DB에서 로드) / logout 시 favorites 비움 / **persist에서 favorites 제외(partialize)** — DB가 진실.- auth/callback에서 로그인 후 loadFavorites 호출.- GameDetail 찜 버튼: 빈 하트 ↔ 빨간 하트(fill-red-500), 텍스트 "찜" 고정, hover 반응, 비로그인 시 **LoginPromptModal**(브라우저 confirm 대신 사이트 톤 모달).- 마이페이지 찜목록 섹션 (카드 hover 시 X 삭제).- 실측: 찜 → DB `user=2 app=...` 저장 확인.**교훈:** 순환 import 걱정 → api.ts가 useUserStore를 dynamic import라 store에서 api 정적 import 안전.

## 2. 회원정보 수정 (온보딩 재활용)- 백엔드 그대로 재활용: `OnboardingView`가 이미 성별/나이대/닉네임 저장 → 수정에도 씀 (onboarding_completed=True 재설정은 무해).- `/mypage/edit` 페이지: 온보딩 폼 복제 + 기존 값 미리 채움(getMe) + 저장 후 마이페이지로 + 취소 버튼.- 이메일은 구글 로그인으로 받은 거라 수정 불가(읽기 전용, 내 정보 카드에만 표시).

## ⭐ 3. 취향 설정 — 선호 장르 + 지표별 점수 (핵심)**준태 그림:** 선호 장르 선택 → 그 장르의 핵심 지표에 좋고싫음 점수 매겨 저장 → (나중에) 추천 반영.**범위 구분 (원칙):** 수집만 지금. 추천 반영(score_v6 user 레이어)은 데이터 쌓인 후. "수집 인프라는 출시 전, 분석 로직은 후."**방식 확정 (B: 장르 연동 동적):** 12개 고정 대신, 고른 장르의 GENRE_CORE_METRICS를 합쳐서(중복 제거, 최대 15개) 그 지표만 물음. 12개는 적다는 준태 지적 → 장르 연동이 정확+본인 의도.**구현:**- CustomUser에 JSONField 2개: `preferred_genres`(['액션','RPG']), `metric_preferences`({'cozy_factor':5}). default list/dict라 기존 유저 안전.- API `TastePreferenceView` (GET/POST 한 view). 지표 점수 1~5 검증.- `/mypage/taste` 페이지: 장르 칩(최대 3개) → 고르면 지표 슬라이더 5단계(별로↔중요) 동적 표시. GENRE_CORE_METRICS 상수 직접 참조(getGenreCoreMetrics는 게임 값 필요해서 취향엔 부적합).- 실측: 액션 선택 → 액션 지표 6개(action_pacing:4, reflex_demand:4, replay_value:2 ...) 저장 확인. = 개인화 추천 재료.**DB 장르 확인:** genres가 콤마 복합 문자열("액션, 어드벤처, 인디"). 개별 장르 7개(액션/어드벤처/RPG/전략/시뮬레이션/캐주얼/인디)로 선택지 구성.

## 4. 회원 탈퇴 — 익명화 (PIPA/GDPR)**방향:** 법적으로 문제없을 정도의 익명화. 개인 식별 정보만 삭제, 통계는 익명으로 보존.- 삭제: CustomUser 레코드(이메일/닉네임/성별/나이/취향), Favorite(개인 취향).- **익명화(SET_NULL로 남김):** UserAction(행동 로그, 이미 SET_NULL), GameSurvey(설문 — CASCADE→SET_NULL로 변경, Community Validation 데이터 보존). user만 NULL → 개인 식별 불가하지만 집계엔 사용 가능.**구현:**- GameSurvey.user: CASCADE → SET_NULL + null=True (migration).- `DeleteAccountView` (DELETE): Favorite 삭제 → user.delete() → UserAction/GameSurvey 자동 익명화.- `DeleteAccountModal`: "탈퇴" 직접 입력해야 버튼 활성(실수 방지), 빨강 경고, 삭제 항목 명시.

## 5. 마이페이지 사이드바 레이아웃**방식 B (라우트 방식):** `/mypage/layout.tsx`(공통 사이드바)로 /mypage·/mypage/taste·/mypage/edit가 레이아웃 공유. Next.js layout.tsx 활용.**구조:**```좌 사이드바:  내 정보 / 취향 설정 / 회원정보 수정 / ─── / 로그아웃 / 회원 탈퇴우 콘텐츠:    각 페이지 (page.tsx는 콘텐츠만, 제목·계정관리는 layout으로)```- 현재 메뉴 보라색 강조(usePathname), 탈퇴는 회색(위험 강조 안 함) + hover 시 빨강.- "설정" 메뉴는 나중 (지금 설정할 항목 없음).**트러블:**- layout.tsx에 실수로 page 내용이 들어감 → 사이드바 안 뜸. layout에 사이드바 코드 다시 넣어 해결.- /mypage/edit 폴더가 실제로 안 만들어져 있었음(404) → 재생성.- page.tsx가 옛날 버전(제목/취향입구/계정관리 중복) → 콘텐츠만 남긴 버전으로 교체.- `.next` 캐시 꼬임(500, survey-test 흔적) → rm -rf .next + dev 재시작.

## 🔀 Project B (데이터 파이프라인) — 별도 세션 진행 중새 Claude 프로젝트로 분리 완료. B가 START_HERE 받고 진단 수행, 우리가 놓친 지뢰 발견:- **"52 vs 60" 규명:** 교사 4,190개는 2-패스(기본 31 + 확장 18)로 만들어짐. batch_generator SYSTEM_PROMPT는 1패스(31)만 → few-shot 개조 시 49+gem으로 확장해 1패스 전량 추출해야.- **적재 블로커 3개:** custom_id 규약 불일치(request- vs game-, 적재 0건), gem_potential 스케일, 스키마 이중화(JSONB vs 정규화 60컬럼).- **models.py가 stale** — game.py엔 gem_percentile·embedding 있는데 models.py엔 없음. **stale models.py로 makemigrations 절대 금지**(embedding DROP 위험). B는 스키마 안 만들고 데이터만 적재 → migration 불필요.- **실측 (A에서 제공):** `SELECT MIN/MAX/AVG(gem_potential)` → 0/100/75.68. **DB 0~100 스케일 확정**, /10 하지 말 것. UPSERT는 gem_percentile·embedding 안 건드리게.- B가 B2B 데이터 상품(3층/베이지안/인구통계 큐브)까지 확장 → 범위 이탈로 판단, 브레이크. 단 "부품 홈 맞추기"(신작 저장 형식이 A game_metrics와 일치) 정합성은 유효. 층 분리는 A가 이미 함(MetricRating our_score vs user_score).

## 📋 다음 할 일**회원 기능 = 완성.** 다음은 배포 방향.**남은 것 (출시 전):**- ⬜ 통합 확인: 수집 인프라(설문/찜/취향)가 실제 유저 흐름에서 잘 도는지- ⬜ 설문 트리거 실확인 (1주일 뒤 자연 발생)- ⬜ 배포: Vercel(프론트) + Railway(백엔드/DB). **pgvector Railway 지원 확인 필수.** 환경변수 이관, CORS 재설정, 백업 cron.**미래 (기억만, 지금 X):**- 로그인 방식 확장: 이메일(코드 송신 방식) + 구글(완료) + 스팀 OpenID- 리뷰 → 포인트 → 구독 할인 시스템- 개인화 추천 (행동/설문/취향 → score_v6 user 레이어) — 데이터 쌓인 후- "설정" 메뉴 (테마 등)- Project B: 신작 파이프라인 (few-shot 개조 → 크롤 → 적재 → percentile 재계산)**잔여:** requirements pytest 영구추가, pydantic Config→ConfigDict.

## 📌 환경 메모- 표준 기동: `docker-compose up -d` + 별터미널 `cd frontend && npm run dev`- Docker Desktop 꺼지면 "cannot find dockerDesktopLinuxEngine" → Docker Desktop 실행 후 `up -d`- `.next` 캐시 꼬임(500/유령 파일 에러): `rm -rf .next` + dev 재시작- bash에서 `!` 들어간 문자열은 `event not found` → sed로 confirm 등 넣을 때 에디터 직접 수정이 안전- Next.js layout.tsx: 폴더에 두면 하위 라우트가 공통 레이아웃 공유- makemigrations는 users 앱만 — games 앱(embedding/gem_percentile) 절대 안 건드리게 확인*— "다 해보자"로 마이페이지 회원 기능을 통째로 끝낸 날. 찜/취향/수정/탈퇴/사이드바. 다음은 배포.*